Install basic dependecies

In [ ]:
%pip install --upgrade langchain-text-splitters langchain-community langgraph tqdm unstructured langgraph langchain-community beautifulsoup4 langchain-huggingface text-generation transformers google-search-results numexpr langchainhub sentencepiece jinja2 bitsandbytes accelerate

In [ ]:
%%capture --no-stderr

Import Langsmith tracing

In [ ]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

HuggingFace API

In [ ]:
import getpass
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass(
    "Enter your Hugging Face API key: "
)

Install HuggingFace

In [ ]:
import torch

In [ ]:
%pip install --upgrade --quiet  langchain-huggingface text-generation transformers google-search-results numexpr langchainhub sentencepiece jinja2 bitsandbytes accelerate

In [ ]:
!pip install --upgrade transformers accelerate

Import Model

In [ ]:

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    endpoint_url="https://aowxr49vng3zvk4q.us-east-1.aws.endpoints.huggingface.cloud",
    task="text-generation",
    max_new_tokens=50000,
    do_sample=False,
    repetition_penalty=1.03,
)

chat_model = ChatHuggingFace(llm=llm)

Import embeddings model

In [ ]:
!pip install -qU langchain-huggingface
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="jinaai/jina-embeddings-v2-base-zh")

Import vector store from Chroma

In [ ]:
pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(embedding_function=embeddings)

Directory Loader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update the DirectoryLoader path to point to the correct location in Google Drive
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader("/content/drive/MyDrive/AI/List", glob="**/*.txt", show_progress=True) # Update this path with the actual location in your Google Drive
docs = loader.load()
len(docs)

Splitting the text

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

Embedding

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

Vector store

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

retriever

In [ ]:
retriever = vector_store.as_retriever()

RAG Fusion

In [ ]:
from langchain.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates 3 search queries from a single user question.
Generate multiple search queries related to：{question} \n
Output（5 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

generate_queries = (
    prompt_rag_fusion
    | ChatHuggingFace(llm=llm)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [ ]:
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60, top_n=8):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents
        and an optional parameter k used in the RRF formula """

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

     # Limit the results to the top n
    reranked_results = reranked_results[:top_n] # Slice the list

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

question = ""
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# RAG
template = """You are an assistant in the question-answering task. Use the following retrieved context to answer the questions. If you don't know the answer, just say you don't know. Don't mention "text" in your answer. The more content the better:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})